# Unlock Your Full Potential as a Business Analyst With the Powerful Causal Impact Framework

**Causal inference can help you become a business analyst rockstar.**

In a business context, the leadership is often interested in the impact of a decision or event on the KPI of interest. As a performance analyst, I spend most of my time answering some variant of this question: “What is the impact of {News, government announcement, special event…} in the Country’s X performance?”. Intuitively, we can answer this question if we had a way of knowing what would have happened if the News/ announcement/ Special event had never happened.

**This is the essence of causal inference, and some very talented people are working hard to make causal inference frameworks available for us to use.**

Google Causal Impact library is one of those frameworks. Developed by Google to help them make better marketing budget decisions, this library can help us quantify the impact of any event or intervention on a time series of interest. It may sound scary, but it's actually quite intuitive.
As business analysts, we should leverage these tools in our day-to-day lives; here are 5 easy steps you can take to implement your first Causal Impact analysis.


In [ ]:
from causalimpact import CausalImpact
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# If data/example.csv is missing, generate it first:
#   python generate_example_data.py
DATA_PATH = Path('data') / 'example.csv'


# Import Data and define pre/post periods

**We can think of the Causal Impact framework as a time series problem.**
On a specific date, we observe an event, news, etc.… and track how our measure of interest changes after this event compared to some baseline. You can think of your baseline as a control group. 

To perform a Causal Impact analysis, we will need the following:

* Date of the event to study
* Time Series of our variable of interest for our impacted unit
* Time Series of our variable of interest for other multiple units that were not impacted by the event

For my example, I am using a made-up scenario as follows:

* Event: Global conference happened on September 6th in Barcelona
* Variable of Interest: Passenger revenue for an airline
* Impacted Unit: Barcelona
* Control group: Other European Markets
* Question: What was the impact of the Conference on Passenger revenue in Barcelona?

When considering the time frame, you should aim to keep the post-period as short as possible, and the pre-period should be longer than your post-period.


In [6]:
# set time periods
training_strat = "2022-03-10"
training_end = "2022-06-30"
treatment_start = "2022-07-01"
treatment_end = "2022-07-13"

In [ ]:
data = pd.read_csv(DATA_PATH, index_col='Date', parse_dates=True)
data.index.freq = 'd'
data.index


In [ ]:
data.head()

# Creating a Control Group

**This step is the most critical part.**

To estimate the impact of any event or treatment, we need to know our unit of interest (example Barcelona Passenger Revenue) under the treatment and under no treatment. This is the main problem of causal inference.

>we can never observe our unit of interest under two mutually exclusive circumstances. The solution is to create a counterfactual scenario.

You can think of a counterfactual as a hypothetical scenario where the event/treatment did not occur.

Our control group will help us create this scenario. To decide which markets to include in our control group, we need to calculate which markets have predictive power to predict our market of interest (Barcelona). We will use correlation to establish this predictive power.
This step needs to be done in order the PRE-TREATMENT  period only.


In [ ]:
# get the training data only
df_training = data[data.index <= pd.to_datetime(training_end)]
df_training.tail()

In [ ]:
# Check  for stationarity
from statsmodels.tsa.stattools import adfuller

test = adfuller(x = df_training['Barcelona'])[1]
print(test)
if test< 0.05:
    print("Data is stationary")
else:
    print("Time series is not stationary")

In [ ]:
# making data stationary by differencing

differencing = df_training.pct_change().dropna(thresh = 1,axis=1).dropna()
differencing.head()

In [ ]:
test = adfuller(x = differencing['Barcelona'])[1]
print(test)
if test< 0.05:
    print("Data is stationary")
else:
    print("Time series is not stationary")

In [ ]:
# Check Top correlatted markets
market_cor= pd.Series(differencing.corr().abs()['Barcelona'])
market_cor[market_cor >=0.3]

In [ ]:
#Markets to keep for model
markets_to_keep = list(market_cor[market_cor >=0.3].index)

markets_to_keep

# Implementing CausalImpact

Now, we are ready to implement CausalImpact.

In a nutshell, CausalImpact will use our control group to learn to predict the Passenger Revenue in Barcelona during the Pre-period. The model will use this to predict the counterfactual post-period scenario where the Conference did not happen.

The Delta between what actually happened and the counterfactual scenario is the impact of the conference.


In [ ]:
#Keep only markets on the above list

final_data = data.drop(columns=[col for col in data if col not in markets_to_keep])

In [ ]:
#Prepare Pre and Post periods
pre_period = [training_strat,training_end]
post_period = [treatment_start,treatment_end]

In [ ]:
impact = CausalImpact(data=final_data,
                      pre_period=pre_period,
                      post_period=post_period)

# Interpreting results & Validation

Google CausalImpact makes it very easy to visualize and summarize the results.


In [ ]:
impact.plot()

In [ ]:
print(impact.summary())

In [ ]:
print(impact.summary('report'))

Depending on what you are measuring, you might be interested in average impact or cumulative impact. In our case, we are interested in the cumulative impact over the period of the conference.

Based on the analysis, the conference contributed to +3M upside in Passenger revenue.

Unlike in machine learning, there are no accuracy measures for Causal Impact, which can make validation a bit tricky. However there are 3 things you can do to validate your results.

1. Ensure your confidence interval in Pre period is not too broad – this could indicate that your control group is not predictive enough
2. Ensure the confidence interval of the estimated impact does not contain 0
3. Use refutation tests, for example, if you conduct the same analysis but change the event date to any day in the pre-period, the impact should be 0
